# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided example for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and examine dataset details using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and print a summary
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Cite as: {getattr(metadata, 'cite_as', getattr(metadata, 'citeAs', 'N/A'))}\n")
print(f"Published: {getattr(metadata, 'date_published', getattr(metadata, 'datePublished', 'N/A'))}")


## 2. Data Overview
Review available record sets, their fields, columns, and `@id` identifiers.

Displaying basic metadata and structure of record sets (tables/collections) described in the dataset.

In [ ]:
# List record sets in the dataset, referencing them by their @id field.
record_sets = dataset.record_sets

print(f"Record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"- RecordSet Name: {rs.name}, @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields (@id):")
    for field in rs.fields:
        print(f"    - {field['name']} (@id: {field['@id']}, type: {field.get('data_type', field.get('dataType', ''))})")
    print()
# For demonstration, also print the first record for each set, by @id:
for rs in record_sets:
    print(f"Preview of first record from record set '@id': {rs.id}")
    records_iter = dataset.records(record_set=rs.id)
    try:
        rec = next(records_iter)
        print(rec)
    except StopIteration:
        print("No records found.")
    print()


## 3. Data Extraction
Load data from a specific record set into a Pandas DataFrame for analysis. You can refer to the list above and select the appropriate `@id` for desired record set(s), field(s), and column(s).


In [ ]:
# Specify the @id(s) of the record set(s) to load (replace with correct @id values from the overview above).
# Example: If the main record set @id is 'senscience:protein_analysis_table' (replace with actual from above output).

record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        # Load all records for the set as a DataFrame
        records = list(dataset.records(record_set=record_set_id))
        if len(records):
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set '@id': {record_set_id}")
            print(f"Columns: {df.columns.tolist()}\n")
        else:
            print(f"No records for record set '@id': {record_set_id}")
    except Exception as e:
        print(f"Failed to load record set '@id': {record_set_id} ({e})")

# For demonstration, select the first record set to preview
if record_set_ids:
    first_record_set_id = record_set_ids[0]
    if first_record_set_id in dataframes:
        print(f"Previewing first 5 rows of '@id': {first_record_set_id}")
        display(dataframes[first_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing: filtering, normalization, and grouping. All references to fields/columns use their `@id` as specified in the Croissant schema.


In [ ]:
### Select a numeric field for analysis
# Inspect columns, and pick a likely numeric field (e.g., peptide_count, MW, normalized_abundance, etc).
selected_record_set = None
for k, v in dataframes.items():
    numeric_columns = v.select_dtypes(include='number').columns.tolist()
    if numeric_columns:
        selected_record_set = k
        break

if selected_record_set is not None:
    numeric_field_id = numeric_columns[0]  # pick the first numeric column by @id (column name)
    print(f"Using record set '@id': {selected_record_set}")
    print(f"Numeric field for demonstration: {numeric_field_id}")
    
    df = dataframes[selected_record_set]
    threshold = df[numeric_field_id].quantile(0.75) if len(df) > 0 else 0  # Use 75th percentile as example cutoff
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with '{numeric_field_id}' > {threshold:.2f} (75th percentile): {len(filtered_df)}")
    display(filtered_df.head())
    
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Attempt grouping by a likely categorical field (choose the first object dtype column not equal to index)
    group_candidates = [col for col in df.columns
                       if col != numeric_field_id and df[col].dtype == 'object']
    group_field = group_candidates[0] if group_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by '{group_field}' and mean of '{numeric_field_id}':")
        display(grouped_df.head())
else:
    print("No record set with numeric columns available for EDA.")


## 5. Visualization
Visualize distributions or relationships for key fields in the data.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()

if selected_record_set is not None and numeric_field_id is not None:
    # Distribution
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[selected_record_set][numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # Grouped means if grouping field found
    if group_field is not None:
        plt.figure(figsize=(12,4))
        plot_df = filtered_df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False).head(20)
        sns.barplot(x=plot_df.values, y=plot_df.index)
        plt.title(f"Top 20 {group_field} by mean {numeric_field_id}")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field)
        plt.show()
else:
    print("No numeric field visualized. Please run the above EDA cell to select valid fields.")

## 6. Conclusion
In this notebook, we have loaded and explored the structure of the FAIR^2 mass spectrometry dataset using standard patterns from the Croissant schema with `mlcroissant`. We:
- Inspected all record sets and their fields using their `@id` identifiers for clarity and reproducibility.
- Loaded one or more record sets into DataFrames for analysis.
- Demonstrated filtering and normalization on a numeric field with explicit `@id` reference.
- Visualized distributions and grouped summaries for selected fields.

You can adapt this notebook further by substituting different record set or field `@id`s for in-depth analysis.

_For more comprehensive feature or advanced machine learning modeling, refer to the [mlcroissant documentation](https://github.com/mlcommons/croissant)._